# NB14 v2 — Normalización por ventana inicial y diagnóstico de errores

**Pregunta:** ¿persiste el beneficio de normalizar si utilizamos solo las primeras 10, 25 o 50 firmas únicas según el índice de origen?

**CPU en Google Colab. Ejecutar todas las celdas en orden.** Usa la carpeta existente `Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1`; crea una carpeta nueva y un ZIP final. No altera NB01–NB13 ni sus predicciones.

Esta es una sensibilidad exploratoria diseñada después de observar resultados externos. No es una validación prospectiva independiente ni una prueba causal. Los índices no son marcas temporales validadas. La eliminación global de firmas compartidas es un control retrospectivo, no una etapa de despliegue.

La comparación RAW/PREFIX utiliza las mismas observaciones dentro de cada ventana. Las ventanas de 10/25/50 pueden producir cohortes distintas: no atribuir toda diferencia entre ventanas al tamaño de calibración. La comparación con estadísticas del lote completo usa el mismo modelo entrenado con prefijos; no constituye una cota superior ni reproduce el pipeline DOMAIN_Z de NB12.

**Cambios:** corrección de `nunique`; semillas emparejadas entre brazos; predicciones individuales; diagnóstico de composición con conjunto de evaluación fijo y separado; figuras de errores por experimento; entorno y procedencia.

**Pendiente:** la incertidumbre por dependencia de mediciones no se resuelve con este notebook. No se añaden intervalos por filas.


# NB14 — Prospective DOMAIN_Z and target-stage-composition sensitivity

**Project:** `BANANA_RIPENESS_MULTISENSOR_Q1`

This notebook examines the concern that experiment-wise DOMAIN_Z used statistics from the complete held-out acquisition trajectory and therefore could exploit future target-domain feature information.

## Primary question

Does the DOMAIN_Z benefit persist when normalization statistics are estimated **only from an initial, pre-specified calibration window** and prediction is evaluated exclusively on subsequent observations?

## Temporal information available in the released data

The released raw CSV files do **not** contain timestamp/date columns. Therefore this notebook does not claim hour/day chronology. It uses the original source acquisition index as the only available order proxy:

- `spect_source_index` when available;
- `ds34_source_index` for Test 4;
- fallback to `tphis_source_index`, then `master_id`.

The primary source-order proxy uses the **first 25 unique spectral signatures** in source order. Sensitivities use 10 and 50 unique signatures.

## Source-order prefix simulation

For each N and each experiment:

1. identify the first N unique spectral signatures in source order;
2. estimate per-channel mean and SD from **only those N unlabeled signatures**;
3. exclude the calibration signatures from model training/evaluation;
4. use only later vector-stage observations;
5. repeat the complete nested LOEO model/hyperparameter selection.

Two predictive arms are compared on the same future-only observations:

- `RAW_FUTURE`: no domain normalization;
- `DOMAIN_Z_PREFIX`: per-channel z-score based only on the first N unique signatures.

A target-only transductive diagnostic comparison is also computed for the selected DOMAIN_Z_PREFIX model:
- `DOMAIN_Z_TARGET_FULL_DIAGNOSTIC`: the same trained model, but the held-out target is normalized using all unique target signatures. This comparator is explicitly non-deployable and quantifies the possible benefit of future target feature information.

## Stage-composition sensitivity

For the primary N=25 policy, a separate **diagnostic** stress test constructs calibration sets of 25 non-conflicting unique target signatures with three early/late ripeness mixtures:
- 80% stages 1–3 / 20% stages 4–5;
- 50% / 50%;
- 20% / 80%.

Target labels are used **only** to construct these stress-test calibration compositions. They are never used in the prospective normalization or in model selection.

## Duplicate/overlap control

As in NB12:
- every exact 18-band signature shared across experiments is removed first;
- exact within-experiment vector-stage repetitions are collapsed;
- conflicting labels for the same spectral vector are retained as distinct vector-stage observations.

**Hardware:** CPU only.


In [ ]:
# 0. Mount Drive, dependencies and configuration
from google.colab import drive
drive.mount('/content/drive')

import sys, subprocess, importlib.util
if importlib.util.find_spec('mord') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mord==0.7'])
if importlib.util.find_spec('xgboost') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, time, warnings, zipfile, re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, mean_absolute_error
)

import mord
import xgboost as xgb

warnings.filterwarnings('ignore')
SEED=2026
np.random.seed(SEED)

PROJECT=Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
MASTER_PATH=PROJECT/'02_EXTRACTED_DATA'/'06_master_original_multisensor.csv'
PROTOCOL_PATH=PROJECT/'04_FROZEN_SPLITS'/'nested_loeo_protocol_v1.json'
RAW_ZIP_DIR=PROJECT/'01_RAW_ZIPS'
RESULTS_ROOT=PROJECT/'06_RESULTS'

assert MASTER_PATH.exists(),MASTER_PATH
assert PROTOCOL_PATH.exists(),PROTOCOL_PATH
assert RAW_ZIP_DIR.exists(),RAW_ZIP_DIR

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR=RESULTS_ROOT/f'NB14_v2_SOURCE_ORDER_DOMAIN_Z_{RUN_ID}'
TABLE_DIR=RUN_DIR/'tables'
FIG_DIR=RUN_DIR/'figures'
TEXT_DIR=RUN_DIR/'text'
META_DIR=RUN_DIR/'metadata'
for d in [TABLE_DIR,FIG_DIR,TEXT_DIR,META_DIR]:
    d.mkdir(parents=True,exist_ok=True)

SPECTRAL_FEATURES=[
    'SpA410','SpB435','SpC460','SpD485','SpE510','SpF535',
    'SpG560','SpH585','SpR610','SpI645','SpS680','SpJ705',
    'SpT730','SpU760','SpV810','SpW860','SpK900','SpL940'
]
ALL_LABELS=[1,2,3,4,5]
OUTER_TESTS=[2,3,4,5]
MODELS=['OrdinalLogistic','PLSDA','SVM_RBF','RandomForest','XGBoost']
WINDOWS=[10,25,50]
PRIMARY_N=25
N_COMP=25
N_COMP_REPS=100
COMP_SCENARIOS={
    'early80_late20':(20,5),
    'balanced50_50':(12,13),
    'early20_late80':(5,20),
}
N_JOBS=max(1,min(8,(os.cpu_count() or 2)-1))

start_all=time.perf_counter()

print('Project:',PROJECT)
print('Output:',RUN_DIR)
print('CPU workers:',N_JOBS)
print('Prospective windows:',WINDOWS,'| primary N=',PRIMARY_N)

from IPython.display import display
import importlib.metadata as metadata
(META_DIR/'installed_packages.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')


In [ ]:
# 1. Frozen provenance and timestamp/order audit
def sha256_file(path,chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_MASTER_SHA='0a06fb3773f85dacee42e18eecf17e5e2a8e7f5e9508cd3d60bd4b66757e8ea8'
EXPECTED_PROTOCOL_SHA='86731ff9a292f1afd35a215105b14c4af3aa2bda5e8c4dad4df639d4d82ad930'

assert sha256_file(MASTER_PATH)==EXPECTED_MASTER_SHA
assert sha256_file(PROTOCOL_PATH)==EXPECTED_PROTOCOL_SHA

timestamp_patterns=re.compile(r'(time|date|timestamp|datetime|hour|minute|second)',re.I)
raw_audit=[]

for zip_name in ['Spect.zip','TPHiS.zip','ds_34.zip']:
    zp=RAW_ZIP_DIR/zip_name
    assert zp.exists(),zp
    with zipfile.ZipFile(zp) as z:
        for member in z.namelist():
            if not member.lower().endswith('.csv') or '_x_' not in member.lower():
                continue
            with z.open(member) as f:
                header=pd.read_csv(f,nrows=0).columns.tolist()
            time_cols=[c for c in header if timestamp_patterns.search(str(c))]
            raw_audit.append({
                'zip':zip_name,
                'member':member,
                'n_columns':len(header),
                'timestamp_like_columns':'|'.join(map(str,time_cols)),
                'has_timestamp_like_column':bool(time_cols),
            })

raw_audit=pd.DataFrame(raw_audit)
raw_audit.to_csv(TABLE_DIR/'01_raw_timestamp_audit.csv',index=False)

assert not raw_audit.has_timestamp_like_column.any(), (
    'A timestamp-like column was found. Inspect before proceeding.'
)

print('Timestamp audit: no timestamp/date columns in raw feature CSVs.')
display(raw_audit)


In [ ]:
# 2. Load original data and construct the only available acquisition-order proxy
df=pd.read_csv(MASTER_PATH,low_memory=False).reset_index(drop=True)
df['test_id']=df['test_id'].astype(int)
df['stage']=df['stage'].astype(int)

assert len(df)==17662
assert df['master_id'].is_unique
assert df[SPECTRAL_FEATURES].notna().all().all()

# Prefer the original Spect source index. Test 4 has no spect_source_index
# because its spectra come from ds_34, so ds34_source_index is used there.
df['acq_order_proxy']=(
    df['spect_source_index']
    .combine_first(df['ds34_source_index'])
    .combine_first(df['tphis_source_index'])
    .combine_first(df['master_id'])
)
assert df.acq_order_proxy.notna().all()

order_audit=[]
for t,g in df.groupby('test_id',sort=True):
    source=(
        'spect_source_index' if g.spect_source_index.notna().any()
        else 'ds34_source_index'
    )
    order_audit.append({
        'test_id':int(t),
        'n_rows':len(g),
        'order_proxy_source':source,
        'min_order':float(g.acq_order_proxy.min()),
        'max_order':float(g.acq_order_proxy.max()),
        'stage_sequence_by_order':'-'.join(
            map(str,g.sort_values('acq_order_proxy').stage.drop_duplicates().tolist())
        )
    })

order_audit=pd.DataFrame(order_audit)
order_audit.to_csv(TABLE_DIR/'02_acquisition_order_proxy_audit.csv',index=False)
display(order_audit)


In [ ]:
# 3. Remove all exact spectral signatures shared across experiments
df['spectral_group_global']=df.groupby(
    SPECTRAL_FEATURES,sort=False,dropna=False
).ngroup()

shared=(df.groupby('spectral_group_global',as_index=False)
        .agg(n_tests=('test_id','nunique'),n_rows=('master_id','size')))
shared_ids=set(shared.loc[shared.n_tests>1,'spectral_group_global'].astype(int))

assert len(shared_ids)==2,shared.loc[shared.n_tests>1]

clean=df.loc[~df.spectral_group_global.isin(shared_ids)].copy()

remaining=(clean.groupby('spectral_group_global').test_id.nunique()>1).sum()
assert int(remaining)==0

# Collapse exact experiment × spectrum × stage repeats, preserving label conflicts.
dedup=(clean.sort_values(['test_id','acq_order_proxy','master_id'])
       .drop_duplicates(
           subset=['test_id']+SPECTRAL_FEATURES+['stage'],
           keep='first'
       )
       .copy()
       .reset_index(drop=True))

assert len(dedup)==1895

# Recompute groups on retained data.
dedup['spectral_group']=dedup.groupby(
    SPECTRAL_FEATURES,sort=False,dropna=False
).ngroup()

print('Rows after shared-signature removal:',len(clean))
print('Vector-stage rows after dedup:',len(dedup))
print('Cross-experiment exact spectral overlap remaining: 0')


In [ ]:
# 4. Build unique-spectrum order table and pre-specify prefix windows
unique_rows=[]
for t,g in clean.sort_values(['test_id','acq_order_proxy','master_id']).groupby('test_id',sort=True):
    # one row per exact spectral vector, keeping first occurrence
    u=(g.drop_duplicates(subset=SPECTRAL_FEATURES,keep='first')
       [['test_id','acq_order_proxy','stage','spectral_group_global']+SPECTRAL_FEATURES]
       .copy())
    # number of distinct stage labels associated with each exact vector
    stage_n=(g.groupby('spectral_group_global').stage.nunique())
    u['n_stages_for_vector']=u.spectral_group_global.map(stage_n).astype(int)
    unique_rows.append(u)

unique_spec=pd.concat(unique_rows,ignore_index=True)
unique_spec=unique_spec.sort_values(['test_id','acq_order_proxy']).reset_index(drop=True)

unique_count=unique_spec.groupby('test_id').size().to_dict()
assert min(unique_count.values())>max(WINDOWS),unique_count

prefix_rows=[]
for t,g in unique_spec.groupby('test_id',sort=True):
    for N in WINDOWS:
        p=g.head(N)
        future=dedup[
            (dedup.test_id.eq(t)) &
            (~dedup.spectral_group_global.isin(set(p.spectral_group_global))) &
            (dedup.acq_order_proxy>p.acq_order_proxy.max())
        ]
        counts=p.stage.value_counts().reindex(ALL_LABELS,fill_value=0)
        fut_counts=future.stage.value_counts().reindex(ALL_LABELS,fill_value=0)
        prefix_rows.append({
            'test_id':int(t),
            'N_unique_prefix':N,
            'prefix_cutoff_order':float(p.acq_order_proxy.max()),
            'future_vector_stage_rows':len(future),
            **{f'prefix_stage_{k}':int(counts[k]) for k in ALL_LABELS},
            **{f'future_stage_{k}':int(fut_counts[k]) for k in ALL_LABELS},
        })

prefix_audit=pd.DataFrame(prefix_rows)
prefix_audit.to_csv(TABLE_DIR/'03_prefix_and_future_stage_composition.csv',index=False)
display(prefix_audit)


In [ ]:
# 5. Precompute prefix/full statistics and future masks for every N and experiment
def stats_from_matrix(X):
    X=np.asarray(X,float)
    mu=X.mean(axis=0)
    sd=X.std(axis=0,ddof=0)
    sd=np.where((~np.isfinite(sd)) | (sd<1e-12),1.0,sd)
    return mu,sd

stats={}
future_masks={}

for N in WINDOWS:
    for t in OUTER_TESTS:
        u=unique_spec[unique_spec.test_id.eq(t)].sort_values('acq_order_proxy')
        prefix=u.head(N)
        mu_p,sd_p=stats_from_matrix(prefix[SPECTRAL_FEATURES].to_numpy(float))
        mu_full,sd_full=stats_from_matrix(u[SPECTRAL_FEATURES].to_numpy(float))

        prefix_groups=set(prefix.spectral_group_global.astype(int))
        cutoff=float(prefix.acq_order_proxy.max())

        mask=(
            dedup.test_id.eq(t) &
            (~dedup.spectral_group_global.isin(prefix_groups)) &
            (dedup.acq_order_proxy>cutoff)
        ).to_numpy()

        assert mask.sum()>0

        stats[(N,t,'prefix')]=(mu_p,sd_p)
        stats[(N,t,'full')]=(mu_full,sd_full)
        future_masks[(N,t)]=mask

print('Prefix/full statistics and future masks ready.')


In [ ]:
# 6. Build train/validation matrices for a given N and arm
X_DEDUP=dedup[SPECTRAL_FEATURES].to_numpy(float)
Y_DEDUP=dedup.stage.to_numpy(int)
T_DEDUP=dedup.test_id.to_numpy(int)

def transform_rows(N,tests,arm,target_full_test=None):
    X_parts=[]
    y_parts=[]
    test_parts=[]

    for t in tests:
        mask=future_masks[(N,t)]
        Xt=X_DEDUP[mask].copy()
        yt=Y_DEDUP[mask].copy()

        if arm=='RAW_FUTURE':
            pass
        elif arm=='DOMAIN_Z_PREFIX':
            kind='full' if (target_full_test is not None and t==target_full_test) else 'prefix'
            mu,sd=stats[(N,t,kind)]
            Xt=(Xt-mu)/sd
        else:
            raise ValueError(arm)

        X_parts.append(Xt)
        y_parts.append(yt)
        test_parts.extend([t]*len(yt))

    return np.vstack(X_parts),np.concatenate(y_parts),np.asarray(test_parts,int)

# Sanity
for N in WINDOWS:
    for t in OUTER_TESTS:
        X,y,_=transform_rows(N,[t],'DOMAIN_Z_PREFIX')
        assert np.isfinite(X).all() and len(X)==len(y)

print('Matrix builder: PASS')


In [ ]:
# 7. Metrics, model implementations and frozen compact grid
def classification_metrics(y_true,y_pred):
    y_true=np.asarray(y_true,int)
    y_pred=np.asarray(y_pred,int)
    observed=sorted(np.unique(y_true).tolist())
    ae=np.abs(y_true-y_pred)
    return {
        'accuracy':float(accuracy_score(y_true,y_pred)),
        'macro_f1_observed':float(f1_score(
            y_true,y_pred,labels=observed,average='macro',zero_division=0
        )),
        'macro_f1_fixed_1to5':float(f1_score(
            y_true,y_pred,labels=ALL_LABELS,average='macro',zero_division=0
        )),
        'balanced_accuracy':float(balanced_accuracy_score(y_true,y_pred)),
        'mcc':float(matthews_corrcoef(y_true,y_pred)),
        'qwk':float(cohen_kappa_score(y_true,y_pred,labels=ALL_LABELS,weights='quadratic')),
        'ordinal_mae':float(mean_absolute_error(y_true,y_pred)),
        'within_1_stage':float(np.mean(ae<=1)),
        'severe_error_ge2':float(np.mean(ae>=2)),
        'observed_labels':','.join(map(str,observed)),
    }

def fit_model(model_name,params,X_train,y_train,seed):
    y_train=np.asarray(y_train,int)

    if model_name=='OrdinalLogistic':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        model=mord.LogisticAT(alpha=float(params['alpha']))
        model.fit(Xt,y_train)
        return ('scaled',model,scaler,None)

    if model_name=='PLSDA':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        classes=np.sort(np.unique(y_train))
        Y=np.zeros((len(y_train),len(classes)),float)
        for j,c in enumerate(classes):
            Y[:,j]=(y_train==c).astype(float)
        ncomp=int(min(params['n_components'],Xt.shape[1],Xt.shape[0]-1))
        model=PLSRegression(n_components=ncomp,scale=False,max_iter=1000)
        model.fit(Xt,Y)
        return ('pls',model,scaler,classes)

    if model_name=='SVM_RBF':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        model=SVC(
            kernel='rbf',
            C=float(params['C']),
            gamma=params['gamma'],
            probability=False,
            cache_size=1000,
            random_state=seed
        )
        model.fit(Xt,y_train)
        return ('scaled',model,scaler,None)

    if model_name=='RandomForest':
        model=RandomForestClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=params['max_depth'],
            min_samples_leaf=int(params['min_samples_leaf']),
            max_features=params['max_features'],
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,y_train)
        return ('plain',model,None,None)

    if model_name=='XGBoost':
        classes=np.sort(np.unique(y_train))
        cmap={c:i for i,c in enumerate(classes)}
        yz=np.array([cmap[c] for c in y_train],int)
        model=xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=len(classes),
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=float(params['learning_rate']),
            subsample=float(params['subsample']),
            colsample_bytree=float(params['colsample_bytree']),
            reg_lambda=1.0,
            min_child_weight=1.0,
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,yz,verbose=False)
        return ('xgb',model,None,classes)

    raise ValueError(model_name)

def predict_model(bundle,X):
    kind,model,scaler,classes=bundle
    if kind=='scaled':
        return np.asarray(model.predict(scaler.transform(X)),int)
    if kind=='pls':
        scores=np.asarray(model.predict(scaler.transform(X)))
        if scores.ndim==1:
            scores=scores[:,None]
        return classes[np.argmax(scores,axis=1)].astype(int)
    if kind=='plain':
        return np.asarray(model.predict(X),int)
    if kind=='xgb':
        return classes[np.asarray(model.predict(X),int)].astype(int)
    raise ValueError(kind)

def candidate_grid(model_name):
    if model_name=='OrdinalLogistic':
        return [{'alpha':0.01},{'alpha':0.1},{'alpha':1.0},{'alpha':10.0}]
    if model_name=='PLSDA':
        return [{'n_components':2},{'n_components':4},{'n_components':8}]
    if model_name=='SVM_RBF':
        return [
            {'C':0.5,'gamma':'scale'},
            {'C':2.0,'gamma':'scale'},
            {'C':8.0,'gamma':'scale'},
            {'C':8.0,'gamma':0.03},
        ]
    if model_name=='RandomForest':
        return [
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':18,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':3,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':0.7},
        ]
    if model_name=='XGBoost':
        return [
            {'n_estimators':250,'max_depth':3,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':350,'max_depth':5,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':250,'max_depth':3,'learning_rate':0.10,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':500,'max_depth':2,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
        ]
    raise ValueError(model_name)

assert sum(len(candidate_grid(m)) for m in MODELS)==19
print('Models and frozen grid ready.')


## Nested LOEO under the prospective policy

For each N, both `RAW_FUTURE` and `DOMAIN_Z_PREFIX` are selected independently using the original inner-LOEO rule.

Expected inner fits:

3 windows × 2 arms × 4 outer folds × 3 inner validation experiments × 19 candidates = **1368 fits**.


In [ ]:
# 8. Full inner LOEO selection
ARMS=['RAW_FUTURE','DOMAIN_Z_PREFIX']

inner_rows=[]
fit_counter=0
inner_start=time.perf_counter()

for N in WINDOWS:
    for arm_idx,arm in enumerate(ARMS):
        for outer_test in OUTER_TESTS:
            dev_tests=[t for t in OUTER_TESTS if t!=outer_test]

            print(f'\nN={N} | {arm} | outer Test {outer_test}')

            for model_idx,model_name in enumerate(MODELS):
                for cand_idx,params in enumerate(candidate_grid(model_name)):
                    pjson=json.dumps(params,sort_keys=True)

                    for inner_val in dev_tests:
                        inner_train=[t for t in dev_tests if t!=inner_val]

                        Xtr,ytr,_=transform_rows(N,inner_train,arm)
                        Xva,yva,_=transform_rows(N,[inner_val],arm)

                        seed=SEED+N*100000+outer_test*100+inner_val*10+cand_idx

                        t0=time.perf_counter()
                        status='ok'; err=''
                        try:
                            bundle=fit_model(model_name,params,Xtr,ytr,seed)
                            pred=predict_model(bundle,Xva)
                            met=classification_metrics(yva,pred)
                        except Exception as e:
                            status='error'; err=repr(e)
                            met={k:np.nan for k in [
                                'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                                'balanced_accuracy','mcc','qwk','ordinal_mae',
                                'within_1_stage','severe_error_ge2'
                            ]}
                            met['observed_labels']=','.join(map(str,sorted(np.unique(yva))))

                        fit_counter+=1
                        inner_rows.append({
                            'N_prefix':N,
                            'arm':arm,
                            'outer_test':outer_test,
                            'inner_val_test':inner_val,
                            'inner_train_tests':','.join(map(str,inner_train)),
                            'model':model_name,
                            'candidate_id':cand_idx,
                            'params_json':pjson,
                            'n_train_future':len(ytr),
                            'n_val_future':len(yva),
                            'status':status,
                            'error':err,
                            'fit_seconds':time.perf_counter()-t0,
                            **met
                        })

                        if fit_counter%100==0:
                            print(f'  {fit_counter}/1368 fits | elapsed {(time.perf_counter()-inner_start)/60:.1f} min')

inner=pd.DataFrame(inner_rows)
inner.to_csv(TABLE_DIR/'04_inner_loeo_prospective_results.csv.gz',index=False,compression='gzip')

assert fit_counter==1368,fit_counter
assert inner.status.eq('ok').all(),inner.loc[~inner.status.eq('ok'),['N_prefix','arm','outer_test','model','error']]

print('\nInner LOEO complete.')
print('Elapsed minutes:',round((time.perf_counter()-inner_start)/60,2))


In [ ]:
# 9. Aggregate candidates and apply original lexicographic selection
agg=(inner.groupby(
    ['N_prefix','arm','outer_test','model','candidate_id','params_json'],
    as_index=False
).agg(
    inner_folds=('inner_val_test','nunique'),
    mean_macro_f1_observed=('macro_f1_observed','mean'),
    sd_macro_f1_observed=('macro_f1_observed','std'),
    worst_macro_f1_observed=('macro_f1_observed','min'),
    mean_macro_f1_fixed_1to5=('macro_f1_fixed_1to5','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_mcc=('mcc','mean'),
    mean_qwk=('qwk','mean'),
    mean_ordinal_mae=('ordinal_mae','mean'),
    mean_severe_error_ge2=('severe_error_ge2','mean'),
    total_fit_seconds=('fit_seconds','sum')
))

assert agg.inner_folds.eq(3).all()

selected_hp=[]
for keys,g in agg.groupby(['N_prefix','arm','outer_test','model'],sort=True):
    gs=g.sort_values(
        ['mean_macro_f1_observed','worst_macro_f1_observed','mean_qwk','mean_ordinal_mae','candidate_id'],
        ascending=[False,False,False,True,True],
        kind='mergesort'
    )
    selected_hp.append(gs.iloc[0].copy())

selected_hp=pd.DataFrame(selected_hp).reset_index(drop=True)
selected_hp.to_csv(TABLE_DIR/'05_selected_hyperparameters_prospective.csv',index=False)

assert len(selected_hp)==3*2*4*5
print('Selected model-specific hyperparameters:',len(selected_hp))


In [ ]:
# 10. Outer LOEO: source-order prefix normalization, RAW comparator, and full-target diagnostic
outer_rows=[]
prediction_rows=[]
outer_fit_counter=0
outer_start=time.perf_counter()

for N in WINDOWS:
    for arm_idx,arm in enumerate(ARMS):
        for outer_test in OUTER_TESTS:
            dev_tests=[t for t in OUTER_TESTS if t!=outer_test]
            Xtr,ytr,_=transform_rows(N,dev_tests,arm)
            Xte,yte,_=transform_rows(N,[outer_test],arm)

            for model_idx,model_name in enumerate(MODELS):
                s=selected_hp[
                    selected_hp.N_prefix.eq(N) &
                    selected_hp.arm.eq(arm) &
                    selected_hp.outer_test.eq(outer_test) &
                    selected_hp.model.eq(model_name)
                ]
                assert len(s)==1
                s=s.iloc[0]
                params=json.loads(s.params_json)

                seed=SEED+N*100000+outer_test*100+model_idx*10
                t0=time.perf_counter()

                bundle=fit_model(model_name,params,Xtr,ytr,seed)
                pred=predict_model(bundle,Xte)
                met=classification_metrics(yte,pred)

                row={
                    'N_prefix':N,
                    'arm':arm,
                    'outer_test':outer_test,
                    'model':model_name,
                    'candidate_id':int(s.candidate_id),
                    'params_json':s.params_json,
                    'inner_mean_macro_f1':float(s.mean_macro_f1_observed),
                    'inner_worst_macro_f1':float(s.worst_macro_f1_observed),
                    'n_train_future':len(ytr),
                    'n_test_future':len(yte),
                    'fit_seconds':time.perf_counter()-t0,
                    **met
                }

                # For DOMAIN_Z_PREFIX only: same trained model, same future-only rows,
                # but normalize the held-out target using all unique target spectra.
                if arm=='DOMAIN_Z_PREFIX':
                    Xte_oracle,yte2,_=transform_rows(
                        N,[outer_test],'DOMAIN_Z_PREFIX',target_full_test=outer_test
                    )
                    assert np.array_equal(yte,yte2)
                    pred_oracle=predict_model(bundle,Xte_oracle)
                    omet=classification_metrics(yte,pred_oracle)
                    for k,v in omet.items():
                        row[f'oracle_{k}']=v
                else:
                    for k in [
                        'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                        'balanced_accuracy','mcc','qwk','ordinal_mae',
                        'within_1_stage','severe_error_ge2','observed_labels'
                    ]:
                        row[f'oracle_{k}']=np.nan

                ids=dedup.loc[future_masks[(N,outer_test)],'master_id'].to_numpy(int)
                assert len(ids)==len(yte) and len(set(ids))==len(ids)
                for k,mid in enumerate(ids):
                    prediction_rows.append({'N_prefix':N,'arm':arm,'outer_test':outer_test,
                        'model':model_name,'master_id':int(mid),'stage_true':int(yte[k]),
                        'stage_pred':int(pred[k]),
                        'full_target_diagnostic_pred':int(pred_oracle[k]) if arm=='DOMAIN_Z_PREFIX' else None})
                pd.DataFrame(prediction_rows).to_csv(TABLE_DIR/'outer_predictions_checkpoint.csv.gz',index=False,compression='gzip')
                outer_rows.append(row)
                outer_fit_counter+=1

                print(
                    f'N={N:2d} | {arm:15s} | T{outer_test} | {model_name:15s} | '
                    f'Macro-F1={met["macro_f1_observed"]:.4f}'
                )

outer=pd.DataFrame(outer_rows)
outer.to_csv(TABLE_DIR/'06_outer_loeo_prospective_all_models.csv',index=False)

assert outer_fit_counter==3*2*4*5
print('\nOuter fits:',outer_fit_counter)
print('Elapsed minutes:',round((time.perf_counter()-outer_start)/60,2))

pd.DataFrame(prediction_rows).to_csv(TABLE_DIR/'outer_predictions_all_models.csv.gz',index=False,compression='gzip')


In [ ]:
# 11. Select the model family by inner performance and summarize descriptive contrast
selected_rows=[]

for keys,g in outer.groupby(['N_prefix','arm','outer_test'],sort=True):
    gs=g.sort_values(
        ['inner_mean_macro_f1','inner_worst_macro_f1','model'],
        ascending=[False,False,True],
        kind='mergesort'
    )
    selected_rows.append(gs.iloc[0].copy())

selected=pd.DataFrame(selected_rows).reset_index(drop=True)
selected.to_csv(TABLE_DIR/'07_selected_outer_prospective.csv',index=False)

# Pair RAW and DOMAIN_Z_PREFIX on the same N/test.
raw_sel=selected[selected.arm.eq('RAW_FUTURE')][[
    'N_prefix','outer_test','model','macro_f1_observed',
    'macro_f1_fixed_1to5','balanced_accuracy','mcc'
]].rename(columns={
    'model':'raw_model',
    'macro_f1_observed':'raw_macro_f1_observed',
    'macro_f1_fixed_1to5':'raw_macro_f1_fixed',
    'balanced_accuracy':'raw_balanced_accuracy',
    'mcc':'raw_mcc',
})

dz_sel=selected[selected.arm.eq('DOMAIN_Z_PREFIX')][[
    'N_prefix','outer_test','model','macro_f1_observed',
    'macro_f1_fixed_1to5','balanced_accuracy','mcc',
    'oracle_macro_f1_observed','oracle_macro_f1_fixed_1to5'
]].rename(columns={
    'model':'dz_model',
    'macro_f1_observed':'dz_prefix_macro_f1_observed',
    'macro_f1_fixed_1to5':'dz_prefix_macro_f1_fixed',
    'balanced_accuracy':'dz_prefix_balanced_accuracy',
    'mcc':'dz_prefix_mcc',
    'oracle_macro_f1_observed':'dz_oracle_macro_f1_observed',
    'oracle_macro_f1_fixed_1to5':'dz_oracle_macro_f1_fixed',
})

paired=raw_sel.merge(dz_sel,on=['N_prefix','outer_test'],validate='one_to_one')
paired['delta_prefix_vs_raw_observed']=paired.dz_prefix_macro_f1_observed-paired.raw_macro_f1_observed
paired['delta_prefix_vs_raw_fixed']=paired.dz_prefix_macro_f1_fixed-paired.raw_macro_f1_fixed
paired['oracle_minus_prefix_observed']=paired.dz_oracle_macro_f1_observed-paired.dz_prefix_macro_f1_observed
paired['oracle_minus_prefix_fixed']=paired.dz_oracle_macro_f1_fixed-paired.dz_prefix_macro_f1_fixed

paired.to_csv(TABLE_DIR/'08_source_order_prefix_vs_raw_and_oracle.csv',index=False)

summary=(paired.groupby('N_prefix',as_index=False)
         .agg(
             mean_raw_macro_f1=('raw_macro_f1_observed','mean'),
             mean_prefix_domainz_macro_f1=('dz_prefix_macro_f1_observed','mean'),
             mean_full_target_oracle_macro_f1=('dz_oracle_macro_f1_observed','mean'),
             mean_delta_prefix_vs_raw=('delta_prefix_vs_raw_observed','mean'),
             median_delta_prefix_vs_raw=('delta_prefix_vs_raw_observed','median'),
             tests_prefix_better_than_raw=('delta_prefix_vs_raw_observed',lambda x:int((np.asarray(x)>0).sum())),
             mean_oracle_minus_prefix=('oracle_minus_prefix_observed','mean'),
             max_abs_oracle_minus_prefix=('oracle_minus_prefix_observed',lambda x:float(np.max(np.abs(x))))
         ))
summary.to_csv(TABLE_DIR/'09_source_order_summary_by_window.csv',index=False)

display(paired.round(4))
print('\nSummary:')
display(summary.round(4))


## Target-stage-composition stress test

This section is intentionally diagnostic rather than deployable. Stage labels are used only to construct calibration subsets with different early/late ripeness mixtures.

The predictive model and its hyperparameters are fixed from the **prospective N=25 inner-LOEO selection**. No retuning is performed for any composition scenario.


In [ ]:
# 12. Stage-composition sensitivity for primary N=25
rng=np.random.default_rng(SEED+1400)
comp_rows=[]
composition_skips=[]

# Spectrum-level table restricted to non-conflicting vectors.
# Determine stage membership only for the diagnostic composition stress test.
group_stage=(clean.groupby(['test_id','spectral_group_global'])
             .stage.agg(['nunique','first']).reset_index()
             .rename(columns={'first':'stage'}))
ucomp=unique_spec.merge(
    group_stage,
    on=['test_id','spectral_group_global'],
    suffixes=('_first','')
)
ucomp=ucomp[ucomp['nunique'].eq(1)].copy()
ucomp['early_late']=np.where(ucomp.stage<=3,'early','late')

# Fixed N=25 DOMAIN_Z-prefix selected model family/hyperparameters per outer fold.
primary=selected[
    (selected.N_prefix.eq(PRIMARY_N)) &
    (selected.arm.eq('DOMAIN_Z_PREFIX'))
].copy()

for outer_test in OUTER_TESTS:
    prow=primary[primary.outer_test.eq(outer_test)]
    assert len(prow)==1
    chosen_model=prow.iloc[0].model

    hp=selected_hp[
        selected_hp.N_prefix.eq(PRIMARY_N) &
        selected_hp.arm.eq('DOMAIN_Z_PREFIX') &
        selected_hp.outer_test.eq(outer_test) &
        selected_hp.model.eq(chosen_model)
    ].iloc[0]
    params=json.loads(hp.params_json)

    dev_tests=[t for t in OUTER_TESTS if t!=outer_test]
    Xtr,ytr,_=transform_rows(PRIMARY_N,dev_tests,'DOMAIN_Z_PREFIX')
    bundle=fit_model(
        chosen_model,params,Xtr,ytr,
        SEED+PRIMARY_N*100000+outer_test*100+777
    )

    tgt=ucomp[ucomp.test_id.eq(outer_test)].copy()
    early=tgt[tgt.early_late.eq('early')]
    late=tgt[tgt.early_late.eq('late')]

    if min(len(early),len(late))<40:
        composition_skips.append({'outer_test':outer_test,'reason':'Fewer than 40 unique nonconflicting signatures in an early/late stratum','n_early':len(early),'n_late':len(late)})
        continue
    # Disjoint pool; target labels are used for this diagnostic stress test only.
    pool_early=early.sample(n=len(early)//2,random_state=SEED+outer_test)
    pool_late=late.sample(n=len(late)//2,random_state=SEED+outer_test+100)
    pool_ids=set(pd.concat([pool_early,pool_late]).spectral_group_global.astype(int))
    fixed_mask=(dedup.test_id.eq(outer_test) & ~dedup.spectral_group_global.isin(pool_ids)).to_numpy()
    fixed_ids=dedup.loc[fixed_mask,'master_id'].to_numpy(int)
    assert fixed_mask.sum()>0
    early,late=pool_early,pool_late

    for scenario,(n_early,n_late) in COMP_SCENARIOS.items():
        for rep in range(N_COMP_REPS):
            es=early.sample(n=n_early,replace=False,random_state=int(rng.integers(1,2**31-1)))
            ls=late.sample(n=n_late,replace=False,random_state=int(rng.integers(1,2**31-1)))
            calib=pd.concat([es,ls],ignore_index=True)

            mu,sd=stats_from_matrix(calib[SPECTRAL_FEATURES].to_numpy(float))
            calib_groups=set(calib.spectral_group_global.astype(int))

            eval_mask=fixed_mask.copy()
            assert not (calib_groups & set(dedup.loc[eval_mask,'spectral_group_global']))

            Xev=(X_DEDUP[eval_mask]-mu)/sd
            yev=Y_DEDUP[eval_mask]
            pred=predict_model(bundle,Xev)
            met=classification_metrics(yev,pred)

            comp_rows.append({
                'outer_test':outer_test,
                'selected_model':chosen_model,
                'scenario':scenario,
                'replicate':rep,
                'n_calibration':N_COMP,
                'n_early':n_early,
                'n_late':n_late,
                'n_eval':len(yev),
                'evaluation_ids_sha256':hashlib.sha256(fixed_ids.tobytes()).hexdigest(),
                **met
            })

composition=pd.DataFrame(comp_rows,columns=['outer_test','selected_model','scenario','replicate','n_calibration','n_early','n_late','n_eval','evaluation_ids_sha256','macro_f1_observed','macro_f1_fixed_1to5','accuracy','balanced_accuracy','mcc','qwk','ordinal_mae','within_1_stage','severe_error_ge2','observed_labels'])
pd.DataFrame(composition_skips,columns=['outer_test','reason','n_early','n_late']).to_csv(TABLE_DIR/'composition_skipped_support.csv',index=False)
composition.to_csv(
    TABLE_DIR/'10_stage_composition_stress_replicates.csv.gz',
    index=False,
    compression='gzip'
)

comp_summary=(composition.groupby(['outer_test','scenario'],as_index=False)
              .agg(
                  mean_macro_f1=('macro_f1_observed','mean'),
                  sd_macro_f1=('macro_f1_observed','std'),
                  median_macro_f1=('macro_f1_observed','median'),
                  q025_macro_f1=('macro_f1_observed',lambda x:float(np.quantile(x,0.025))),
                  q975_macro_f1=('macro_f1_observed',lambda x:float(np.quantile(x,0.975))),
                  mean_fixed_macro_f1=('macro_f1_fixed_1to5','mean')
              ))
comp_summary.to_csv(TABLE_DIR/'11_stage_composition_stress_summary.csv',index=False)

display(comp_summary.round(4))


In [ ]:
# 13. Figures

# A. Source-order prefix vs RAW vs full-target oracle
fig,axes=plt.subplots(1,3,figsize=(15,4.6),sharey=True)
for ax,N in zip(axes,WINDOWS):
    g=paired[paired.N_prefix.eq(N)].sort_values('outer_test')
    ax.plot(g.outer_test,g.raw_macro_f1_observed,marker='o',label='RAW future-only')
    ax.plot(g.outer_test,g.dz_prefix_macro_f1_observed,marker='s',label='DOMAIN_Z prefix')
    ax.plot(g.outer_test,g.dz_oracle_macro_f1_observed,marker='^',linestyle='--',
            label='Full-target diagnostic')
    ax.set_xticks(OUTER_TESTS)
    ax.set_xlabel('Held-out experiment')
    ax.set_title(f'First {N} unique spectra')
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Observed-class Macro-F1')
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=3,bbox_to_anchor=(0.5,-0.04))
fig.suptitle('Prospective domain normalization using only an initial target window',y=1.02)
fig.tight_layout(rect=[0,0.08,1,1])

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2I_prospective_domainz_by_window.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)

# B. Oracle-minus-prefix gap
fig,ax=plt.subplots(figsize=(8.5,5))
for N,marker in zip(WINDOWS,['o','s','^']):
    g=paired[paired.N_prefix.eq(N)].sort_values('outer_test')
    ax.plot(
        g.outer_test,g.oracle_minus_prefix_observed,
        marker=marker,linewidth=2,label=f'N={N}'
    )
ax.axhline(0,linestyle='--',linewidth=1)
ax.set_xticks(OUTER_TESTS)
ax.set_xlabel('Held-out experiment')
ax.set_ylabel('Macro-F1: full-target oracle − prefix normalization')
ax.set_title('Potential benefit of future target feature information')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2J_future_information_gap.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)

# C. Composition stress: boxplots by test
fig,axes=plt.subplots(2,2,figsize=(11,8),sharey=False)
axes=axes.ravel()
scenario_order=list(COMP_SCENARIOS.keys())

for ax,t in zip(axes,OUTER_TESTS):
    g=composition[composition.outer_test.eq(t)]
    if g.empty:
        ax.text(.5,.5,'Insufficient support for fixed-cohort stress test',ha='center',va='center',transform=ax.transAxes,wrap=True)
        ax.set_title(f'Test {t}');continue
    data=[
        g[g.scenario.eq(s)].macro_f1_observed.to_numpy()
        for s in scenario_order
    ]
    ax.boxplot(data,labels=['80/20','50/50','20/80'],showfliers=False)
    ax.set_title(f'Test {t}')
    ax.set_xlabel('Early (stages 1–3) / Late (4–5)')
    ax.set_ylabel('Macro-F1')
    ax.grid(axis='y',alpha=0.2)

fig.suptitle('Sensitivity of DOMAIN_Z to calibration-stage composition',y=1.01)
fig.tight_layout()

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2K_stage_composition_sensitivity.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)


In [ ]:
# 14. Headline results and interpretation
primary_paired=paired[paired.N_prefix.eq(PRIMARY_N)].copy()
primary_summary=summary[summary.N_prefix.eq(PRIMARY_N)].iloc[0]

# Exact two-sided sign-test resolution for n=4 if all four differences have same sign.
min_two_sided_sign_p=2*(0.5**4)

key={
    'timestamps_available_in_raw_features':False,
    'order_proxy':'spect_source_index; ds34_source_index for Test 4; fallback tphis_source_index/master_id',
    'windows_unique_spectra':WINDOWS,
    'primary_window_unique_spectra':PRIMARY_N,
    'primary_mean_raw_future_macro_f1':float(primary_summary.mean_raw_macro_f1),
    'primary_mean_domainz_prefix_macro_f1':float(primary_summary.mean_prefix_domainz_macro_f1),
    'primary_mean_target_full_oracle_macro_f1':float(primary_summary.mean_full_target_oracle_macro_f1),
    'primary_mean_delta_prefix_vs_raw':float(primary_summary.mean_delta_prefix_vs_raw),
    'primary_tests_prefix_better_than_raw':int(primary_summary.tests_prefix_better_than_raw),
    'primary_mean_oracle_minus_prefix':float(primary_summary.mean_oracle_minus_prefix),
    'primary_max_abs_oracle_minus_prefix':float(primary_summary.max_abs_oracle_minus_prefix),
    'minimum_two_sided_exact_sign_p_for_4_of_4':float(min_two_sided_sign_p),
    'per_test_primary':primary_paired.to_dict('records'),
    'composition_stress':comp_summary.to_dict('records'),
}

(TABLE_DIR/'12_key_results_nb14.json').write_text(
    json.dumps(key,indent=2,ensure_ascii=False,default=str),
    encoding='utf-8'
)

notes=[
    '# NB14 — prospective DOMAIN_Z analysis',
    '',
    '## Data chronology',
    '- No timestamp/date fields exist in the released raw feature CSV files.',
    '- Therefore, this analysis uses the original source index as an acquisition-order proxy and does not claim hour/day timing.',
    f'- Primary calibration window: first {PRIMARY_N} unique spectral signatures; sensitivity windows: {WINDOWS}.',
    '',
    '## Primary N=25 result',
    f'- Mean RAW future-only Macro-F1: {primary_summary.mean_raw_macro_f1:.4f}.',
    f'- Mean prospective DOMAIN_Z Macro-F1: {primary_summary.mean_prefix_domainz_macro_f1:.4f}.',
    f'- Mean full-target diagnostic Macro-F1 on the same future-only rows: {primary_summary.mean_full_target_oracle_macro_f1:.4f}.',
    f'- Mean prospective DOMAIN_Z minus RAW: {primary_summary.mean_delta_prefix_vs_raw:+.4f}.',
    f'- DOMAIN_Z better than RAW in {int(primary_summary.tests_prefix_better_than_raw)}/4 outer experiments.',
    f'- Mean oracle minus prospective prefix gap: {primary_summary.mean_oracle_minus_prefix:+.4f}.',
    f'- Maximum absolute fold-specific oracle-prefix gap: {primary_summary.max_abs_oracle_minus_prefix:.4f}.',
    f'- With only four outer experiments, the minimum possible two-sided exact sign-test p for 4/4 changes in one direction is {min_two_sided_sign_p:.3f}.',
    '',
    '## Interpretation guardrail',
    'If prefix-only DOMAIN_Z remains materially above RAW while the full-target diagnostic adds little, the normalization result is robust to future-target feature leakage under the available order proxy.',
    'If the full-target diagnostic substantially exceeds prefix-only DOMAIN_Z, the earlier whole-target result should be described as optimistic/transductive and removed from the headline.',
    '',
    '## Composition stress',
    'The early/late calibration-mixture analysis is diagnostic only: target stage labels are used to construct stress-test calibration subsets and are not part of the deployable prospective pipeline.',
]

(TEXT_DIR/'nb14_prospective_domainz_notes.md').write_text(
    '\n'.join(notes)+'\n',encoding='utf-8'
)
print('\n'.join(notes))


In [ ]:
# Diagnóstico pareado de errores: modelos elegidos exclusivamente por validación interna
predictions=pd.DataFrame(prediction_rows)
keys=['N_prefix','arm','outer_test','model']
chosen=predictions.merge(selected[keys],on=keys,how='inner',validate='many_to_one')
chosen.to_csv(TABLE_DIR/'selected_predictions_paired_diagnostics.csv.gz',index=False,compression='gzip')
index=['N_prefix','outer_test','master_id','stage_true']
wide=chosen.pivot(index=index,columns='arm',values='stage_pred').reset_index()
assert wide[['RAW_FUTURE','DOMAIN_Z_PREFIX']].notna().all().all()
wide['raw_abs_error']=(wide.RAW_FUTURE-wide.stage_true).abs()
wide['prefix_abs_error']=(wide.DOMAIN_Z_PREFIX-wide.stage_true).abs()
wide['delta_abs_error']=wide.prefix_abs_error-wide.raw_abs_error
wide['outcome']=np.select([wide.delta_abs_error<0,wide.delta_abs_error>0],['Improved','Worsened'],default='Unchanged')
wide.to_csv(TABLE_DIR/'paired_error_outcomes.csv.gz',index=False,compression='gzip')
primary_errors=wide[wide.N_prefix.eq(PRIMARY_N)].copy()
records=[]
for t,g in primary_errors.groupby('outer_test'):
    records.append({'outer_test':t,'n':len(g),'raw_ordinal_mae':g.raw_abs_error.mean(),
      'prefix_ordinal_mae':g.prefix_abs_error.mean(),'delta_ordinal_mae':g.delta_abs_error.mean(),
      'raw_severe_pct':100*g.raw_abs_error.ge(2).mean(),'prefix_severe_pct':100*g.prefix_abs_error.ge(2).mean(),
      **{k:100*g.outcome.eq(k).mean() for k in ['Improved','Unchanged','Worsened']}})
error_summary=pd.DataFrame(records)
error_summary.to_csv(TABLE_DIR/'paired_error_summary_by_experiment.csv',index=False)
# Same observations, ordinal distances; no row-wise confidence intervals.
plt.rcParams.update({'font.size':11,'pdf.fonttype':42,'svg.fonttype':'none'})
fig,axes=plt.subplots(1,3,figsize=(16,4.8))
for e,col,label in [('raw_abs_error','#777777','RAW'),('prefix_abs_error','#0072B2','Prefix normalization')]:
    # Give each experiment equal weight in the overview.
    dist=primary_errors.groupby('outer_test')[e].value_counts(normalize=True).unstack(fill_value=0).reindex(columns=range(5),fill_value=0).mean()*100
    axes[0].plot(range(5),dist,marker='o',color=col,label=label)
axes[0].set(xlabel='Absolute stage error',ylabel='Experiment-balanced observations (%)',xticks=range(5),title='A. Ordinal error distribution');axes[0].legend(frameon=False)
left=np.zeros(len(error_summary))
for k,color in [('Improved','#0072B2'),('Unchanged','#999999'),('Worsened','#D55E00')]:
    axes[1].barh(error_summary.outer_test.astype(str),error_summary[k],left=left,color=color,label=k)
    left+=error_summary[k].to_numpy()
axes[1].set(xlabel='Observations (%)',ylabel='Held-out experiment',xlim=(0,100),title='B. Paired error outcomes');axes[1].legend(frameon=False,fontsize=9)
axes[2].plot(error_summary.outer_test,error_summary.delta_ordinal_mae,marker='o',color='#0072B2')
axes[2].axhline(0,color='black',ls='--',lw=1)
axes[2].set(xlabel='Held-out experiment',ylabel='MAE: prefix minus RAW (stages)',xticks=OUTER_TESTS,title='C. Error change by experiment')
fig.suptitle('Source-order prefix sensitivity | N=25 unique spectra',fontsize=14)
fig.tight_layout()
for ext in ['png','pdf','svg']:fig.savefig(FIG_DIR/f'Figure_NB14_Paired_Error_Diagnostics.{ext}',dpi=600,bbox_inches='tight')
plt.show();plt.close(fig)
display(error_summary)


In [ ]:
# 15. Manifest, hashes, ZIP and automatic download
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

elapsed=time.perf_counter()-start_all

manifest={
    'project':'BANANA_RIPENESS_MULTISENSOR_Q1',
    'notebook':'14_PROSPECTIVE_DOMAIN_Z_TEMPORAL_PROXY',
    'run_id':RUN_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'hardware':'CPU',
    'seed':SEED,
    'timestamp_available':False,
    'temporal_proxy':'original source acquisition index',
    'windows':WINDOWS,
    'primary_N':PRIMARY_N,
    'composition_calibration_N':N_COMP,
    'composition_replicates_per_test_scenario':N_COMP_REPS,
    'inner_fits':int(fit_counter),
    'outer_fits':int(outer_fit_counter),
    'shared_exact_spectral_signatures_removed':int(len(shared_ids)),
    'vector_stage_rows_after_dedup':int(len(dedup)),
    'key_results':key,
    'versions':{
        'python':platform.python_version(),
        'numpy':np.__version__,
        'pandas':pd.__version__,
        'scikit_learn':sklearn.__version__,
        'xgboost':xgb.__version__,
    },
    'elapsed_minutes':elapsed/60,
}
(META_DIR/'run_manifest.json').write_text(
    json.dumps(manifest,indent=2,ensure_ascii=False,default=str),
    encoding='utf-8'
)

hash_rows=[]
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file() and p.name!='package_hashes.csv':
        hash_rows.append({
            'relative_path':str(p.relative_to(RUN_DIR)),
            'bytes':p.stat().st_size,
            'sha256':file_sha(p)
        })
pd.DataFrame(hash_rows).to_csv(META_DIR/'package_hashes.csv',index=False)

readme=[
    '# NB14 Prospective DOMAIN_Z',
    '',
    f'Run ID: `{RUN_ID}`',
    f'Runtime: {elapsed/60:.2f} min',
    '',
    '- No raw timestamps were available.',
    '- Temporal proxy: original source acquisition index.',
    '- Primary prefix: first 25 unique spectra.',
    '- Sensitivities: first 10 and 50 unique spectra.',
    '- Full nested LOEO repeated for RAW future-only and prefix-only DOMAIN_Z.',
    '- Same trained target model also evaluated with full-target statistics as a transductive diagnostic.',
    '- Stage-composition stress test included.',
    '',
    'Upload the generated ZIP to ChatGPT for interpretation.'
]
(RUN_DIR/'README_RESULTS.md').write_text(
    '\n'.join(readme)+'\n',encoding='utf-8'
)

zip_base=RESULTS_ROOT/f'RESULTS_NB14_PROSPECTIVE_DOMAIN_Z_{RUN_ID}'
zip_path=Path(shutil.make_archive(
    str(zip_base),'zip',
    root_dir=RUN_DIR.parent,
    base_dir=RUN_DIR.name
))

latest={
    'notebook':'14_PROSPECTIVE_DOMAIN_Z_TEMPORAL_PROXY',
    'run_id':RUN_ID,
    'run_dir':str(RUN_DIR),
    'zip_path':str(zip_path),
    'zip_bytes':zip_path.stat().st_size,
    'elapsed_minutes':elapsed/60,
}
(RESULTS_ROOT/'LATEST_NB14_PROSPECTIVE_DOMAIN_Z_RUN.json').write_text(
    json.dumps(latest,indent=2),
    encoding='utf-8'
)

print('\n✅ NB14 COMPLETED')
print('Run directory:',RUN_DIR)
print('ZIP:',zip_path)
print('ZIP size (MB):',round(zip_path.stat().st_size/1024**2,2))
print('Elapsed minutes:',round(elapsed/60,2))
print('\nUpload this ZIP to ChatGPT.')

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Automatic browser download did not complete:',repr(e))
    print('ZIP remains safely stored in Drive:',zip_path)
